# L08 – Hyperparametrar och learning rate schedules

**Tid:** ca 2 h · **Del B: Träna ett nätverk väl**

## Mål
1. Veta vilka hyperparametrar som är **viktigast** och i vilken ordning de ska justeras.
2. Hitta en bra **learning rate** med ett enkelt svep på log-skala.
3. Göra en enkel **random search** och förstå varför den slår grid search.
4. Använda två **LR schedules**, `StepLR` och `ReduceLROnPlateau`, och veta varför man sänker learning rate under träningen.

**Var i arbetsflödet?** Steg 8. Alla val görs mot **valideringsmängden**. Testmängden rör vi inte.

> Kräver `dlkit` från L07. I den här lektionen får `dlkit.fit` stöd för schedulers.

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()

def make_mlp(hidden=256, dropout=0.0):
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden // 2, 10))

## 1. Vilka hyperparametrar spelar roll?

Ett neuralt nät har många rattar, men de är inte lika viktiga. En ungefärlig prioriteringsordning:

| Prioritet | Hyperparameter | Kommentar |
|---|---|---|
| 1 | **Learning rate** | Den viktigaste i nästan alla fall. Fel med en faktor 10 kan avgöra om träningen fungerar eller inte. |
| 2 | **Antal epoker** | Sköts av early stopping (L07). |
| 3 | **Modellstorlek** (bredd och djup) | Öka vid underanpassning. |
| 4 | **Regularisering** (weight decay, dropout) | Öka vid överanpassning. |
| 5 | **Batchstorlek** | Bestäms ofta av GPU-minnet. Påverkar vilken learning rate som är bäst. |
| 6 | Optimerarens detaljer (betas, momentum) | Standardvärdena fungerar nästan alltid. |

**Praktiskt:** börja med rimliga standardval (Adam/AdamW, `lr=1e-3`, batch 128, early stopping). Justera sedan learning rate, och därefter storlek och regularisering utifrån diagnosen i L06.

För att experimenten ska gå snabbt använder vi **10 000** träningsbilder. Det är vanligt att söka på en mindre datamängd eller under färre epoker, och att sedan träna den bästa konfigurationen fullt ut.

In [ ]:
train_loader, val_loader, test_loader = fashion_mnist_loaders(train_subset=10_000)
from dlkit.train import fit

## 2. Learning rate-svep på log-skala

Learning rate varierar över **storleksordningar**, så vi testar den på log-skala: 1e-5, 1e-4, 1e-3 och så vidare. Att testa 0,001, 0,002 och 0,003 är meningslöst innan du vet vilken storleksordning som gäller.

In [ ]:
lrs = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
sweep = {}
for lr in lrs:
    set_seed(42)
    model = make_mlp().to(device)
    h = fit(model, train_loader, val_loader, loss_fn,
            torch.optim.Adam(model.parameters(), lr=lr), device, epochs=5, verbose=False)
    sweep[lr] = h
    print(f"lr={lr:<7g} bästa val loss {min(h['val_loss']):.3f}  bästa val acc {max(h['val_acc']):.3f}")

plt.figure(figsize=(7, 4))
plt.semilogx(lrs, [min(h["val_loss"]) for h in sweep.values()], "o-")
plt.xlabel("learning rate (log-skala)"); plt.ylabel("bästa val loss"); plt.grid(alpha=0.3)
plt.title("Learning rate-svep (Adam, 5 epoker)"); plt.show()

Kurvan har den typiska **U-formen**. För låg learning rate gör att modellen inte hinner lära sig, och för hög gör träningen instabil. För Adam hamnar optimum oftast nära **1e-3**. Därför är det standardvärdet. Om du vill finjustera, testa sedan några värden runt det bästa, till exempel 3e-4, 1e-3 och 3e-3.

## 3. Batchstorlek och learning rate hänger ihop

**Gissa först:** vi tränar 5 epoker med samma learning rate men olika batchstorlek. Vilken batchstorlek blir bäst, och vilken är snabbast?

In [ ]:
for bs in [16, 128, 1024]:
    set_seed(42)
    tl, vl, _ = fashion_mnist_loaders(batch_size=bs, train_subset=10_000)
    model = make_mlp().to(device)
    t0 = time.time()
    h = fit(model, tl, vl, loss_fn, torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=5, verbose=False)
    print(f"batch {bs:5d}: {len(tl):4d} steg/epok  bästa val acc {max(h['val_acc']):.3f}  tid {time.time()-t0:.1f}s")

Med en stor batch tas få steg per epok. Med samma learning rate hinner modellen inte lika långt. Väldigt små batcher ger många brusiga steg och utnyttjar inte GPU:n väl. **Tumregel:** välj en batchstorlek mellan 32 och 256 som får plats i minnet. Om du ändrar den mycket, testa learning rate igen.

## 4. Random search

När flera hyperparametrar ska väljas samtidigt är **random search** ett bra standardval. Det motsvarar `RandomizedSearchCV` i sklearn.

**Varför inte grid search (`GridSearchCV`)?** Med ett rutnät på 3 × 3 testas bara **3 olika värden** av den viktigaste parametern, eftersom resten av försöken upprepar samma värden. Med 9 slumpmässiga försök testas **9 olika värden** av varje parameter. Eftersom bara några få parametrar brukar spela stor roll hittar random search bra värden med färre försök (Bergstra och Bengio, 2012).

**Log-uniform fördelning** för parametrar som spänner över storleksordningar (learning rate, weight decay): dra exponenten likformigt, så att 1e-4 till 1e-3 får lika stor chans som 1e-3 till 1e-2.

In [ ]:
rng = np.random.default_rng(42)

def sample_config(rng):
    return {
        "lr": 10 ** rng.uniform(-4, -2),            # log-uniform mellan 1e-4 och 1e-2
        "weight_decay": 10 ** rng.uniform(-4, -1),  # log-uniform mellan 1e-4 och 1e-1
        "hidden": int(rng.choice([128, 256, 512])),
        "dropout": float(rng.uniform(0.0, 0.5)),
    }

trials = []
t0 = time.time()
for trial in range(10):
    cfg = sample_config(rng)
    set_seed(42)
    model = make_mlp(cfg["hidden"], cfg["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    h = fit(model, train_loader, val_loader, loss_fn, opt, device, epochs=8, patience=3, verbose=False)
    trials.append({**cfg, "val_loss": min(h["val_loss"]), "val_acc": max(h["val_acc"]), "epochs": len(h["val_loss"])})
    print(f"försök {trial+1:2d}: val acc {trials[-1]['val_acc']:.3f}")

df = pd.DataFrame(trials).sort_values("val_loss")
print(f"\ntotal tid {time.time()-t0:.0f}s")
df.round(4)

Titta på tabellen. Vilka parametrar skiljer de bästa försöken från de sämsta? Oftast är det learning rate. En spridningsgraf gör det tydligare:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, col in zip(axes, ["lr", "weight_decay", "hidden", "dropout"]):
    ax.scatter(df[col], df["val_loss"])
    if col in ("lr", "weight_decay"):
        ax.set_xscale("log")
    ax.set_xlabel(col); ax.grid(alpha=0.3)
axes[0].set_ylabel("bästa val loss"); plt.tight_layout(); plt.show()

Med bara 10 försök är det här en grov bild. I verkliga projekt kör man fler försök, och verktyg som *Optuna* kan avbryta dåliga försök tidigt (se "Senare fördjupning" i `plan.md`). **Principen är densamma:** slumpa konfigurationer, jämför på val och välj den bästa.

## 5. Learning rate schedules

**Varför ändra learning rate under träningen?** I början är modellen långt från ett bra läge, och stora steg tar oss snabbt framåt. Senare är vi nära, och stora steg gör att vi "studsar" runt minimum. En lägre learning rate mot slutet låter modellen finjustera.

| Schedule | Hur | När |
|---|---|---|
| `StepLR(step_size, gamma)` | multiplicera med `gamma` var `step_size`:e epok | när du ungefär vet hur många epoker träningen tar |
| `ReduceLROnPlateau(factor, patience)` | sänk med `factor` när val loss slutat förbättras | när du inte vet. Den anpassar sig efter kurvan. Ett robust standardval. |

Vi bygger in schedulers i `dlkit.fit`. Det nya är markerat med `# NEW (L08)`.

In [ ]:
%%writefile ../dlkit/train.py
import copy
import time

import torch


def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """One pass over the training data. Returns (mean loss, accuracy)."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    """Evaluate without gradients. Returns (mean loss, accuracy)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def predict(model, loader, device):
    """Return (y_true, y_pred, probs) as NumPy arrays – ready for sklearn.metrics."""
    model.eval()
    ys, probs = [], []
    for X, y in loader:
        logits = model(X.to(device))
        probs.append(torch.softmax(logits, dim=1).cpu())
        ys.append(y)
    probs = torch.cat(probs)
    return torch.cat(ys).numpy(), probs.argmax(dim=1).numpy(), probs.numpy()


def fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=10,
        patience=None, scheduler=None, verbose=True):
    """Train for up to `epochs` epochs. Returns a history dict.

    patience:  early stopping – stop after this many epochs without val-loss
               improvement and restore the best weights (L07).
    scheduler: a torch.optim.lr_scheduler, stepped once per epoch.          # NEW (L08)
               ReduceLROnPlateau gets the val loss.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]  # NEW (L08): lr used this epoch
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr)):
            history[key].append(value)

        if scheduler is not None:  # NEW (L08)
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()

        improved = va_loss < best_loss
        if improved:
            best_loss, best_epoch, bad_epochs = va_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1

        if verbose:
            print(f"epok {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f} | lr {lr:.1e} | {time.time() - t0:.1f}s"
                  + ("  *" if improved else ""))

        if patience is not None and bad_epochs >= patience:
            if verbose:
                print(f"Early stopping: ingen förbättring på {patience} epoker.")
            break

    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print(f"Återställde vikterna från epok {best_epoch} (val loss {best_loss:.4f}).")
    history["best_epoch"] = best_epoch
    return history

In [ ]:
import importlib, dlkit.train
importlib.reload(dlkit.train)
from dlkit.train import fit

schedules = {
    "konstant": lambda opt: None,
    "StepLR(5, 0.1)": lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.1),
    "ReduceLROnPlateau(0.5, 2)": lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2),
}
sched_results = {}
for name, make_sched in schedules.items():
    set_seed(42)
    model = make_mlp(256, 0.2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
    sched_results[name] = fit(model, train_loader, val_loader, loss_fn, opt, device,
                              epochs=15, scheduler=make_sched(opt), verbose=False)
    h = sched_results[name]
    print(f"{name:26s} bästa val loss {min(h['val_loss']):.3f}  bästa val acc {max(h['val_acc']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in sched_results.items():
    axes[0].plot(range(1, 16), h["lr"], "o-", label=name)
    axes[1].plot(range(1, 16), h["val_loss"], "o-", label=name)
axes[0].set_yscale("log"); axes[0].set_title("learning rate per epok")
axes[1].set_title("val loss per epok")
for ax in axes: ax.set_xlabel("epok"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Vad vi ser:**
- När learning rate sänks sjunker val loss ofta ett snäpp direkt. Bruset från de stora stegen försvinner.
- `StepLR` sänker vid fasta epoker, oavsett hur det går.
- `ReduceLROnPlateau` sänker först när kurvan planat ut. Kombinera den med early stopping och låt `patience` för early stopping vara **större** än schedulerns, så att schedulern hinner sänka innan träningen avbryts.

## 6. Övningar

### Övning 1 – log-uniform sampling
Skriv `sample_log_uniform(low, high, rng)` som returnerar ett tal mellan `low` och `high`, där exponenten (i bas 10) är likformigt fördelad.

In [ ]:
def sample_log_uniform(low, high, rng):
    # TODO: använd np.log10 och rng.uniform
    ...

In [ ]:
r = np.random.default_rng(0)
samples = np.array([sample_log_uniform(1e-5, 1e-1, r) for _ in range(20_000)])
assert samples.min() >= 1e-5 and samples.max() <= 1e-1
assert abs(np.median(samples) - 1e-3) / 1e-3 < 0.1, "medianen ska ligga nära det geometriska medelvärdet 1e-3"
assert abs((samples < 1e-4).mean() - 0.25) < 0.02, "varje storleksordning ska få ca 25 %"
print("Rätt!")

### Övning 2 – utöka sökningen
Lägg till **batchstorlek** (32, 64, 128 eller 256) i `sample_config` och kör 8 försök. Kommer batchstorleken med bland de viktigaste parametrarna? Skriv din tolkning.

*Ledtråd:* du behöver skapa nya loaders i varje försök med `fashion_mnist_loaders(batch_size=..., train_subset=10_000)`.

In [ ]:
# TODO

*Din tolkning:*

### Övning 3 – träna den bästa konfigurationen fullt ut
Ta den bästa konfigurationen från random search (`df.iloc[0]`) och träna den på **alla 50 000** bilder med `ReduceLROnPlateau` och early stopping (högst 30 epoker). Utvärdera på **testmängden en gång** till sist.

In [ ]:
best = df.iloc[0]
# TODO
final_test_acc = None

In [ ]:
assert final_test_acc > 0.87, final_test_acc
print("Rätt! test accuracy:", round(final_test_acc, 3))

## När använder jag detta?
- **LR-svep på log-skala:** i början av varje nytt problem eller när du bytt optimerare eller batchstorlek.
- **Random search:** när du ska välja 3 eller fler hyperparametrar. Gör den på en mindre datamängd eller under färre epoker, och träna sedan den bästa konfigurationen fullt ut.
- **Grid search:** bara för 1–2 parametrar med få diskreta värden.
- **`ReduceLROnPlateau`:** ett robust standardval tillsammans med early stopping.
- **`StepLR`:** när du har en fast träningsbudget och vet ungefär hur kurvan ser ut.

## Sammanfattning
- Justera learning rate först, sedan modellstorlek och regularisering. Early stopping sköter antalet epoker.
- Sök på log-skala för parametrar som spänner över storleksordningar.
- Random search hittar bra värden med färre försök än grid search.
- En sänkt learning rate mot slutet ger ofta en sista förbättring.
- `dlkit.fit` har nu `scheduler=` och loggar `lr` per epok.

**Nästa lektion:** spara och ladda modeller, logga experiment och felsöka träning som inte fungerar.